# <font color='red'>SECTION 1: ENVIRONMENT SETUP & PUBLICATION STANDARDS</font>
---
**Purpose:** Set up reproducible, publication-quality analysis environment  
**Outputs:** Configured environment, helper functions, directory structure

**1.1 CORE LIBRARIES**

In [1]:
import warnings
warnings.filterwarnings('ignore') # Hide non-critical warnings to keep notebook output clean

# Data processing
import numpy as np
import pandas as pd
from pathlib import Path # File paths and configuration formats
import json
import yaml

from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import squarify # pip install squarify

# Statistical analysis
# Statistical tests used during EDA and validation
from scipy import stats
from scipy.stats import ks_2samp, chi2_contingency, mannwhitneyu

# Fairness & Ethics
# Metrics for basic demographic fairness evaluation
try:
    from fairlearn.metrics import demographic_parity_ratio, demographic_parity_difference
    print("Fairlearn available")
except ImportError: # Fairness analysis is optional and environment-dependent
    print("Fairlearn not installed - install with: pip install fairlearn --break-system-packages")

# Advanced visualizations
import math
try:
    from mpl_toolkits.mplot3d import Axes3D
    print("3D plotting available")
except:
    print("3D plotting not available")

# External Recommendation Tools (optional)
# Used only for benchmarking or reference experiments
try:
    from recommenders.datasets import movielens
    from recommenders.datasets.python_splitters import python_stratified_split
    print("Microsoft Recommenders available")
except:
    print("Microsoft Recommenders not installed (optional)")

# Environment check
print("\n" + "="*80)
print("LIBRARIES LOADED")
print("="*80)



Fairlearn available
3D plotting available
Microsoft Recommenders not installed (optional)

LIBRARIES LOADED


**1.2 REPRODUCIBILITY SETTINGS**

In [2]:
# Fix random seed to ensure consistent and repeatable results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Basic run metadata for traceability
print(f"\nRandom seed set to: {RANDOM_SEED}")
print(f"Analysis run date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Researcher        : Nazlı Özgür")
print(f"Institution       : Istanbul University - MIS")
print(f"Project           : XAE-Frame v3.6")



Random seed set to: 42
Analysis run date : 2026-01-06 13:42:12
Researcher        : Nazlı Özgür
Institution       : Istanbul University - MIS
Project           : XAE-Frame v3.6


**1.3 PUBLICATION-QUALITY VISUALIZATION STANDARDS**

In [3]:
# Configure global plotting settings for consistent and readable figures

print("\n" + "="*80)
print("CONFIGURING PUBLICATION STANDARDS")
print("="*80)

# Figure quality (IEEE/ACM standard)
# Higher DPI for saved figures, comfortable size for notebook display
plt.rcParams['figure.dpi'] = 150 # Notebook display
plt.rcParams['savefig.dpi'] = 300 # Publication save
plt.rcParams['figure.figsize'] = (12, 6)

# Font hierarchy (IEEE standard)
# Simple font hierarchy to improve readability across plots
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['font.family'] = 'serif'

# Grid and style
# Light grid for easier value tracking without visual clutter
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Seaborn theme
# Keep a clean and consistent style across all figures
sns.set_style("whitegrid")

print("Figure DPI: 150 (display) / 300 (save)")
print("Font family: serif (IEEE standard)")
print("Grid style: whitegrid (alpha=0.3)")


CONFIGURING PUBLICATION STANDARDS
Figure DPI: 150 (display) / 300 (save)
Font family: serif (IEEE standard)
Grid style: whitegrid (alpha=0.3)


**1.4 COLOR-BLIND SAFE PALETTES (Nature Machine Intelligence Compliant)**

In [4]:
# Define color-blind friendly palettes for consistent and accessible visualizations

# Color-blind safe palettes
CB_COLORS = {
    # Sequential (single hue) - useful for densities and distributions
    'sequential_blue': 'Blues',
    'sequential_green': 'Greens',
    'sequential_viridis': 'viridis',  # Perceptually uniform
    'sequential_cividis': 'cividis',  # Designed for color-vision deficiencies
    
    # Diverging (two hues) - for correlations
    'diverging_rdbu': 'RdBu',         # Red-Blue
    'diverging_rdylbu': 'RdYlBu',     # Red-Yellow-Blue
    'diverging_coolwarm': 'coolwarm', # Blue-Red (use with caution)
    
    # Categorical (distinct colors) - color-blind safe colors
    'categorical_safe': ['#0173B2', '#DE8F05', '#029E73', '#CC78BC',
                         '#CA9161', '#FBAFE4', '#949494', '#ECE133'],
    
    # Recommended defaults
    'default': 'viridis',
    'heatmap_safe': 'cividis',
}

# Set default palette
sns.set_palette('viridis')

# Configuration summary

print("\nColor-blind safe palettes configured:")
print(f"Default: {CB_COLORS['default']}")
print(f"Heatmap: {CB_COLORS['heatmap_safe']}")
print("Avoiding 'coolwarm' unless necessary (color-blind issues)")



Color-blind safe palettes configured:
Default: viridis
Heatmap: cividis
Avoiding 'coolwarm' unless necessary (color-blind issues)


**1.5 HELPER FUNCTIONS (Publication Quality)**

In [5]:
# Utility functions used across EDA and reporting for consistent figures

def save_publication_figure(fig, filename, formats=['png', 'svg'], dpi=300):
    """
    Save figure in multiple publication-quality formats.
    
    Args:
        fig: matplotlib figure object
        filename: base filename (without extension)
        formats: list of formats to save ['png', 'svg', 'pdf']
        dpi: resolution for raster formats
        
    Example:
        fig, ax = plt.subplots()
        ax.plot([1, 2, 3])
        save_publication_figure(fig, '01_example_plot')
    """
    output_dir = Path('docs/figures/eda')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for fmt in formats:
        filepath = output_dir / f"{filename}.{fmt}"
        fig.savefig(filepath, dpi=dpi, bbox_inches='tight', format=fmt)
        print(f"Saved: {filepath}")

def add_statistical_annotation(ax, x, y, n, p_value=None, effect_size=None):
    """
    Add statistical annotations to plot (n, p-value, effect size).
    
    Args:
        ax: matplotlib axis
        x, y: position for annotation
        n: sample size
        p_value: p-value from statistical test
        effect_size: effect size (Cohen's d, etc.)
        
    Example:
        fig, ax = plt.subplots()
        ax.scatter([1, 2, 3], [4, 5, 6])
        add_statistical_annotation(ax, 0.5, 0.9, n=100, p_value=0.001)
    """
    text = f"n={n}"
    if p_value is not None:
        if p_value < 0.001:
            text += f"\np<0.001***"
        elif p_value < 0.01:
            text += f"\np={p_value:.3f}**"
        elif p_value < 0.05:
            text += f"\np={p_value:.3f}*"
        else:
            text += f"\np={p_value:.3f}"
    
    if effect_size is not None:
        text += f"\nd={effect_size:.2f}"
    
    ax.text(x, y, text, fontsize=9, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
            transform=ax.transAxes)

def add_panel_labels(axes, labels=['A', 'B', 'C', 'D', 'E', 'F']):
    """
    Add panel labels (A, B, C, D) to subplots for publication.
    
    Args:
        axes: array of matplotlib axes or single axis
        labels: list of labels
        
    Example:
        fig, axes = plt.subplots(2, 2)
        add_panel_labels(axes.flatten())
    """
    # Handle single axis
    if not hasattr(axes, '__len__'):
        axes = [axes]
    
    # Flatten if needed
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else axes
    
    for ax, label in zip(axes_flat, labels):
        ax.text(-0.1, 1.1, label, transform=ax.transAxes,
                fontsize=16, fontweight='bold', va='top', ha='right')

def create_correlation_heatmap(df, method='pearson', title='Correlation Heatmap',
                               figsize=(12, 10), cmap='cividis'):
    """
    Create publication-quality correlation heatmap.
    
    Args:
        df: DataFrame with numerical columns
        method: 'pearson', 'spearman', or 'kendall'
        title: plot title
        figsize: figure size
        cmap: colormap (use color-blind safe!)
        
    Returns:
        fig, ax: matplotlib figure and axis
        
    Example:
        fig, ax = create_correlation_heatmap(df[num_cols])
    """
    # Calculate correlation
    corr = df.corr(method=method)
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Heatmap
    sns.heatmap(corr,
                annot=True,     # Show values
                fmt=".2f",      # 2 decimal places
                cmap=cmap,      # Color-blind safe
                center=0,       # Center at 0
                square=True,    # Square cells
                linewidths=0.5, # Cell borders
                cbar_kws={'label': f'{method.capitalize()} Correlation'},
                ax=ax)
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    return fig, ax

print("\nHelper functions loaded:")
print(" - save_publication_figure")
print(" - add_statistical_annotation")
print(" - add_panel_labels")
print(" - create_correlation_heatmap")




Helper functions loaded:
 - save_publication_figure
 - add_statistical_annotation
 - add_panel_labels
 - create_correlation_heatmap


**1.6 DIRECTORY STRUCTURE**

In [6]:
# Create required project directories if they do not already exist

print("\n" + "="*80)
print("CREATING DIRECTORY STRUCTURE")
print("="*80)

# List of directories used throughout the project

directories = [
    'data/raw',           # Original, unmodified data
    'data/processed',     # Cleaned and transformed datasets
    'docs/figures/eda',   # EDA figures
    'reports/eda',        # EDA summaries and outputs
    'reports/models',     # Model evaluation reports
    'models/checkpoints', # Saved model artifacts
    'logs',               # Experiment and run logs
]

for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {directory}")


CREATING DIRECTORY STRUCTURE
Created directory: data/raw
Created directory: data/processed
Created directory: docs/figures/eda
Created directory: reports/eda
Created directory: reports/models
Created directory: models/checkpoints
Created directory: logs


**1.7 CONFIGURATION EXPORT (for reproducibility)**

In [7]:
# Persist key settings and metadata for reproducibility and traceability

config = {
    'metadata': {
        'project': 'XAE-Frame v3.6',
        'researcher': 'Nazlı Özgür',
        'institution': 'Istanbul University - MIS',
        'date': datetime.now().isoformat(),
        'random_seed': RANDOM_SEED
    },
    'visualization': {
        'dpi_display': 150,
        'dpi_save': 300,
        'default_palette': CB_COLORS['default'],
        'heatmap_palette': CB_COLORS['heatmap_safe'],
        'font_family': 'serif'
    },
    'standards': {
        'reproducibility': 'NIST IR 8312',
        'publication': 'IEEE/ACM',
        'color_blind': 'Nature Machine Intelligence',
        'fairness': 'EU AI Act + Fairlearn'
    }
}

# Export configuration
with open('reports/eda/eda_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("\nConfiguration exported: reports/eda/eda_config.json")


Configuration exported: reports/eda/eda_config.json


**<font color='green'>SECTION 1 COMPLETE & CHECKPOINT: Verify environment</font>**

In [8]:
print("\n" + "="*80)
print("SECTION 1: ENVIRONMENT SETUP COMPLETE")
print("="*80)

# Quick test
test_data = pd.DataFrame({
    'A': np.random.randn(100),
    'B': np.random.randn(100)
})

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(test_data['A'], test_data['B'], alpha=0.5)
ax.set_title('Environment Test Plot', fontweight='bold')
add_panel_labels(ax, ['A'])
plt.close()  # Don't display, just test

print("Environment sanity check completed")
print("Proceeding to Section 2: Data Loading & Basic Profiling")
print("=" * 80 + "\n")


SECTION 1: ENVIRONMENT SETUP COMPLETE
Environment sanity check completed
Proceeding to Section 2: Data Loading & Basic Profiling



# <font color='red'>SECTION 2: DATA LOADING & BASIC PROFILING</font>
---
**Purpose:** Load multiple domain datasets and understand structure  
**Outputs:** Combined dataset loaded, domain-wise profile displayed

**2.1 DATA LOADING**

In [9]:
# Load datasets from different domains to support cross-domain analysis
# Define categories to analyze
CATEGORIES = [
    'All_Beauty',   # Personal care, subjective preferences
    'Electronics',  # Technical, objective specifications
    'Books'         # Content-based, narrative focus
]

print(f"\nLoading {len(CATEGORIES)} domains: {', '.join(CATEGORIES)}")


Loading 3 domains: All_Beauty, Electronics, Books


In [10]:
# Column names (Amazon Reviews 2023 schema-actual column names in JSONL)
# Explicit column references are used to avoid hard-coded strings later

COL_USER = 'user_id' # or 'reviewer_id' or 'reviewerID'
COL_ITEM = 'asin' # Amazon Standard Identification Number
COL_RATING = 'rating' # 1-5 stars
COL_TIMESTAMP = 'timestamp' # Unix timestamp or date string
COL_VERIFIED = 'verified_purchase' # or 'verified' or 'verified_purchase'
COL_HELPFUL = 'helpful_vote' # or 'helpful' or 'vote'

**2.2 LOAD EACH DOMAIN SEPARATELY**

In [ ]:
# Load domain-specific datasets and keep them isolated for later comparison

dfs_dict = {}   # Dictionary to store each domain's DataFrame
load_stats = [] # Track loading statistics

for category in CATEGORIES:
    # Construct file path for JSONL
    filepath = f'../data/raw/{category}.jsonl'

    print(f"\nLoading {category}...")
    print(f"Path: {filepath}")
    
    try:
        # Load JSONL file
        data = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError:
                    continue  # Skip malformed lines

        # Convert to DataFrame
        df_temp = pd.DataFrame(data)
        
        # Add explicit domain identifier
        df_temp['domain'] = category     
        
        # Store in dictionary
        dfs_dict[category] = df_temp
        
        # Calculate statistics
        n_rows = len(df_temp)
        n_cols = df_temp.shape[1]
        memory_mb = df_temp.memory_usage(deep=True).sum() / 1024**2   
        
        # Display
        print(f"Loaded successfully!")
        print(f"Rows: {n_rows:,}")
        print(f"Columns: {n_cols}")
        print(f"Memory: {memory_mb:.2f} MB")
        print(f"Date range: {df_temp[COL_TIMESTAMP].min()} to {df_temp[COL_TIMESTAMP].max()}" 
              if COL_TIMESTAMP in df_temp.columns else "No timestamp column")

      # Store stats
        load_stats.append({
            'domain': category,
            'rows': n_rows,
            'columns': n_cols,
            'memory_mb': memory_mb,
            'status': 'SUCCESS'
        })
        
    except FileNotFoundError:
        print(f"File not found!")
        print(f"Expected path: {filepath}")
        
        load_stats.append({
            'domain': category,
            'rows': 0,
            'columns': 0,
            'memory_mb': 0,
            'status': 'FILE_NOT_FOUND'
        })
        
    except Exception as e:
        print(f"Error loading {category}: {str(e)}")
        
        load_stats.append({
            'domain': category,
            'rows': 0,
            'columns': 0,
            'memory_mb': 0,
            'status': f'ERROR: {str(e)}'
        })

# Check if any domains loaded successfully
if len(dfs_dict) == 0:
    print("\nNo datasets loaded! Check file paths and formats.")
    raise FileNotFoundError("No datasets found.")
else:
    print(f"\nSuccessfully loaded {len(dfs_dict)} domains")


# CHECK ACTUAL COLUMN NAMES
print("\n" + "="*80)
print("CHECKING ACTUAL COLUMN NAMES")
print("="*80)

# Check first domain's columns
first_domain = list(dfs_dict.keys())[0]
sample_df = dfs_dict[first_domain]

print(f"\nColumns in {first_domain}:")
print(sample_df.columns.tolist())

print(f"\nSample row:")
print(sample_df.head(1).to_dict('records')[0])

print("\nIMPORTANT: Update COL_* variables in Section 0 if needed!")
print(" Check if column names match:")
print(f"   - user_id? → {COL_USER}")
print(f"   - asin? → {COL_ITEM}")
print(f"   - rating? → {COL_RATING}")
print(f"   - timestamp? → {COL_TIMESTAMP}")
print(f"   - verified_purchase? → {COL_VERIFIED}")
print(f"   - helpful_vote? → {COL_HELPFUL}")


Loading All_Beauty...
Path: ../data/raw/All_Beauty.jsonl


**2.3 COMBINE ALL DOMAINS & COMPREHENSIVE DIMENSIONS**

In [ ]:
# Merge all domains into a single DataFrame and report dataset dimensions

if len(dfs_dict) == 0:
    raise FileNotFoundError("No datasets found. Please check data sources.")

# Concatenate all DataFrames
df = pd.concat(dfs_dict.values(), ignore_index=True)

# Combined dimensions
n_rows, n_cols = df.shape
total_memory_mb = df.memory_usage(deep=True).sum() / 1024**2
avg_bytes_per_row = (total_memory_mb * 1024**2) / n_rows

global_summary = pd.DataFrame([{
    "Rows": f"{n_rows:,}",
    "Columns": n_cols,
    "Memory_MB": f"{total_memory_mb:.2f}",
    "Avg_Bytes_per_Row": f"{avg_bytes_per_row:.1f}",
    "Domains": df["domain"].nunique()
}])

print("\n" + "=" * 80)
print("COMBINED DIMENSIONS:")
print("=" * 80 + "\n")
display(global_summary)

# Domain distribution
domain_dist_df = (
    df["domain"]
    .value_counts()
    .rename_axis("Domain")
    .reset_index(name="Rows")
)

domain_dist_df["Row_%"] = (
    domain_dist_df["Rows"] / n_rows * 100
).round(1)

print("\n" + "=" * 80)
print("DISTRIBUTION BY DOMAIN:")
print("=" * 80 + "\n")
display(domain_dist_df)

# Per-domain dimension summary
dimension_summary = []

for domain in CATEGORIES:
    df_domain = df[df["domain"] == domain]
    if df_domain.empty:
        continue

    rows = len(df_domain)
    mem_mb = df_domain.memory_usage(deep=True).sum() / 1024**2
    avg_bytes = (mem_mb * 1024**2) / rows

    # Date range (if available)
    if COL_TIMESTAMP in df_domain.columns:
        df_domain[COL_TIMESTAMP] = pd.to_datetime(df_domain[COL_TIMESTAMP])
        min_date = df_domain[COL_TIMESTAMP].min()
        max_date = df_domain[COL_TIMESTAMP].max()
        span_days = (max_date - min_date).days
    else:
        min_date, max_date, span_days = None, None, None

    dimension_summary.append({
        "Domain": domain,
        "Rows": rows,
        "Row_%": round(rows / n_rows * 100, 1),
        "Memory_MB": round(mem_mb, 2),
        "Avg_Bytes_per_Row": round(avg_bytes, 1),
        "Start_Date": min_date,
        "End_Date": max_date,
        "Time_Span_Days": span_days
    })


dimension_df = pd.DataFrame(dimension_summary)

print("\n" + "=" * 80)
print("PER-DOMAIN DIMENSION SUMMARY:")
print("=" * 80 + "\n")
display(dimension_df)

# Persist outputs

dimension_df.to_csv("reports/eda/01_dimension_analysis.csv", index=False)
load_stats_df = pd.DataFrame(load_stats)

print("\n" + "=" * 80)
print("LOADING SUMMARY:")
print("=" * 80 + "\n")
display(load_stats_df)


**2.4 BASIC PROFILING**

In [ ]:
# Structure and data types
df.info()

**2.5 COLUMN TYPE IDENTIFICATION**

In [ ]:
# Identify numerical, categorical and datetime features for downstream analysis

# Identify column types
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()            # Numeric
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist() # Categorical
datetime_cols = df.select_dtypes(include=['datetime64']).columns.tolist()    # Datetime

# Create column type summary table
column_type_summary = []

for col in df.columns:
    col_type = df[col].dtype
    
    # Determine category
    if col in num_cols:
        category = 'Numerical'
    elif col in cat_cols:
        category = 'Categorical'
    elif col in datetime_cols:
        category = 'Datetime'
    else:
        category = 'Other'
    
    # Get unique count
    unique_count = df[col].nunique()
    unique_ratio = unique_count / len(df)
    
    # Get null count
    null_count = df[col].isnull().sum()
    null_pct = (null_count / len(df)) * 100
    
    column_type_summary.append({
        'Column': col,
        'Type': f"{category}",
        'DType': str(col_type),
        'Unique': f"{unique_count:,}",
        'Unique_%': f"{unique_ratio:.1%}",
        'Nulls': f"{null_count:,}",
        'Null_%': f"{null_pct:.1f}%"
    })

# Create DataFrame
column_types_df = pd.DataFrame(column_type_summary)

print("\n" + "="*80)
print("COLUMN TYPE SUMMARY TABLE:")
print("="*80 + "\n")
display(column_types_df)

# Summary Statistics

print("\n" + "="*80)
print("COLUMN TYPE STATISTICS")
print("="*80)

type_counts = pd.DataFrame({
    'Type': ['Numerical', 'Categorical', 'Datetime'],
    'Count': [len(num_cols), len(cat_cols), len(datetime_cols)],
    'Percentage': [
        f"{len(num_cols)/len(df.columns)*100:.1f}%",
        f"{len(cat_cols)/len(df.columns)*100:.1f}%",
        f"{len(datetime_cols)/len(df.columns)*100:.1f}%"
    ]
})

print("\n")
display(type_counts)

# Export Column Type Analysis

column_types_df.to_csv('reports/eda/01_column_types.csv', index=False)
print("Saved: reports/eda/01_column_types.csv")


**2.6 DESCRIPTIVE STATISTICS (Overall and By Domain)**

In [ ]:
# Compute summary statistics for numerical features (overall and by domain)

print("\n" + "="*80)
print("DESCRIPTIVE STATISTICS - OVERALL")
print("="*80 + "\n")

# Overall numerical summary
describe_overall = df[num_cols].describe().T
display(describe_overall)

# Export overall statistics
describe_overall.to_csv('reports/eda/01_descriptive_statistics_overall.csv')
print("Saved: reports/eda/01_descriptive_statistics_overall.csv")

# Domain-wise statistics
print("\n" + "="*80)
print("DESCRIPTIVE STATISTICS - BY DOMAIN")
print("="*80)

for domain in CATEGORIES:
    if domain in df['domain'].values:
        df_domain = df[df['domain'] == domain]
        
        print(f"\n{domain.upper()}:")
        print(f"Rows: {len(df_domain):,}")
        
        describe_domain = df_domain[num_cols].describe().T
        display(describe_domain)
        
        # Export domain statistics
        describe_domain.to_csv(f'reports/eda/01_descriptive_statistics_{domain}.csv')
        print(f"\nSaved: reports/eda/01_descriptive_statistics_{domain}.csv")

**2.7 FIRST LOOK AT DATA**

In [ ]:
# Inspect a small sample of rows to validate data structure and content

print("\n" + "="*80)
print("FIRST 5 ROWS FROM EACH DOMAIN")
print("="*80)

for domain in CATEGORIES:
    if domain in df['domain'].values:
        df_domain = df[df['domain'] == domain]
        print(f"\n{domain.upper()} - First 5 rows:")
        display(df_domain.head(5))

print("\n" + "="*80)
print("RANDOM 10 ROWS (MIXED DOMAINS)")
print("="*80 + "\n")

display(df.sample(10, random_state=42))

**2.8 VALUE COUNTS FOR KEY CATEGORICAL COLUMNS**

In [ ]:
# Summarize rating and verified purchase distributions (overall and by domain)

print("\n" + "="*80)
print("VALUE COUNTS - KEY CATEGORICAL COLUMNS")
print("="*80)

# Rating Distribution - Comprehensive Table

if COL_RATING in df.columns:
    print("\nRATING DISTRIBUTION - OVERALL & BY DOMAIN")
    print("="*80 + "\n")
    
    # Prepare rating distribution data
    rating_dist_data = []
    
    # Overall ratings
    overall_ratings = df[COL_RATING].value_counts().sort_index()
    rating_row = {'Domain': 'OVERALL'}
    for rating in [1, 2, 3, 4, 5]:
        count = overall_ratings.get(rating, 0)
        pct = (count / len(df)) * 100
        rating_row[f'{rating}'] = f"{count:,} ({pct:.1f}%)"
    rating_row['Mean'] = f"{df[COL_RATING].mean():.2f}"
    rating_row['Median'] = f"{df[COL_RATING].median():.1f}"
    rating_row['Std'] = f"{df[COL_RATING].std():.2f}"
    rating_dist_data.append(rating_row)
    
    # Per-domain ratings
    for domain in CATEGORIES:
        if domain in df['domain'].values:
            df_domain = df[df['domain'] == domain]
            domain_ratings = df_domain[COL_RATING].value_counts().sort_index()
            
            rating_row = {'Domain': domain}
            for rating in [1, 2, 3, 4, 5]:
                count = domain_ratings.get(rating, 0)
                pct = (count / len(df_domain)) * 100
                rating_row[f'{rating}'] = f"{count:,} ({pct:.1f}%)"
            rating_row['Mean'] = f"{df_domain[COL_RATING].mean():.2f}"
            rating_row['Median'] = f"{df_domain[COL_RATING].median():.1f}"
            rating_row['Std'] = f"{df_domain[COL_RATING].std():.2f}"
            rating_dist_data.append(rating_row)
    
    # Create and display table
    rating_dist_df = pd.DataFrame(rating_dist_data)
    display(rating_dist_df)
    
    # Export
    rating_dist_df.to_csv('reports/eda/01_rating_distribution.csv', index=False)
    print("Saved: reports/eda/01_rating_distribution.csv")

# Verified Purchase- Comprehensive Table
# Verified purchase distribution (overall and by domain)

if COL_VERIFIED in df.columns:
    print("\n" + "="*80)
    print("VERIFIED PURCHASE - OVERALL & BY DOMAIN")
    print("="*80 + "\n")
    
    # Prepare verified purchase data
    verified_data = []
    
    # Overall verified
    total_verified = df[COL_VERIFIED].sum()
    total_unverified = len(df) - total_verified
    verified_pct = (total_verified / len(df)) * 100
    unverified_pct = (total_unverified / len(df)) * 100
    
    verified_data.append({
        'Domain': 'OVERALL',
        'Total_Reviews': f"{len(df):,}",
        'Verified': f"{total_verified:,}",
        'Verified_%': f"{verified_pct:.1f}%",
        'Unverified': f"{total_unverified:,}",
        'Unverified_%': f"{unverified_pct:.1f}%"
    })
    
    # Per-domain verified
    for domain in CATEGORIES:
        if domain in df['domain'].values:
            df_domain = df[df['domain'] == domain]
            
            domain_verified = df_domain[COL_VERIFIED].sum()
            domain_unverified = len(df_domain) - domain_verified
            domain_verified_pct = (domain_verified / len(df_domain)) * 100
            domain_unverified_pct = (domain_unverified / len(df_domain)) * 100
            
            verified_data.append({
                'Domain': domain,
                'Total_Reviews': f"{len(df_domain):,}",
                'Verified': f"{domain_verified:,}",
                'Verified_%': f"{domain_verified_pct:.1f}%",
                'Unverified': f"{domain_unverified:,}",
                'Unverified_%': f"{domain_unverified_pct:.1f}%"
            })
    
    # Create and display table
    verified_df = pd.DataFrame(verified_data)
    display(verified_df)
    
    # Export
    verified_df.to_csv('reports/eda/01_verified_purchase.csv', index=False)
    print("Saved: reports/eda/01_verified_purchase.csv")

# Key Insights Summary

print("\n" + "="*80)
print("KEY INSIGHTS FROM VALUE COUNTS")
print("="*80)

insights = []

if COL_RATING in df.columns:
    # Rating insights
    mean_rating = df[COL_RATING].mean()
    mode_rating = df[COL_RATING].mode()[0]
    
    insights.append(f"Overall mean rating: {mean_rating:.2f}/5.0")
    insights.append(f"Most common rating: {mode_rating}")
    
    # Rating distribution
    rating_5_pct = (df[COL_RATING] == 5).sum() / len(df) * 100
    rating_1_pct = (df[COL_RATING] == 1).sum() / len(df) * 100
    
    if rating_5_pct > 50:
        insights.append(f"Highly positive: {rating_5_pct:.1f}% are 5-star reviews")
    if rating_1_pct > 10:
        insights.append(f"Significant negativity: {rating_1_pct:.1f}% are 1-star reviews")

if COL_VERIFIED in df.columns:
    verified_overall_pct = (df[COL_VERIFIED].sum() / len(df)) * 100
    insights.append(f"Verified purchases: {verified_overall_pct:.1f}% of all reviews")
    
    if verified_overall_pct < 50:
        insights.append(f"Low verification rate - potential trust issues")
    elif verified_overall_pct > 80:
        insights.append(f"High verification rate - trustworthy dataset")

# Domain comparison
if COL_RATING in df.columns and len(CATEGORIES) > 1:
    domain_means = {}
    for domain in CATEGORIES:
        if domain in df['domain'].values:
            domain_means[domain] = df[df['domain'] == domain][COL_RATING].mean()
    
    best_domain = max(domain_means, key=domain_means.get)
    worst_domain = min(domain_means, key=domain_means.get)
    
    insights.append(f"Highest rated domain: {best_domain} ({domain_means[best_domain]:.2f})")
    insights.append(f"Lowest rated domain: {worst_domain} ({domain_means[worst_domain]:.2f})")

# Display insights
print("\n")
for insight in insights:
    print(f"{insight}")

print("\n" + "="*80)

**2.9 CARDINALITY ANALYSIS**

In [ ]:
# Analyze unique value counts to identify high-cardinality features and domain-level entity sizes (users, items)

print("\n" + "="*80)
print("CARDINALITY ANALYSIS - OVERALL")
print("="*80 + "\n")

# Overall cardinality (column-level)

cardinality_df = (
    pd.DataFrame({
        "Column": df.columns,
        "Unique_Count": [df[col].nunique() for col in df.columns],
        "Unique_Ratio": [df[col].nunique() / len(df) for col in df.columns],
    })
    .sort_values("Unique_Count", ascending=False)
    .reset_index(drop=True)

)

# Flag high-cardinality columns (>50%)
cardinality_df["High_Cardinality"] = cardinality_df["Unique_Ratio"] > 0.5

print("\nOverall Column Cardinality:")
display(cardinality_df)


# Domain-level cardinality metrics (users, items, verification)
 
print("\n" + "="*80)
print("KEY CARDINALITY METRICS - BY DOMAIN")
print("="*80)

cardinality_by_domain = []

domain_cardinality = []

for domain in CATEGORIES:
    df_domain = df[df["domain"] == domain]
    if df_domain.empty:
        continue

    user_count = df_domain[COL_USER].nunique() if COL_USER in df_domain.columns else None
    item_count = df_domain[COL_ITEM].nunique() if COL_ITEM in df_domain.columns else None
    verified_pct = (
        round(df_domain[COL_VERIFIED].mean() * 100, 1)
        if COL_VERIFIED in df_domain.columns
        else None
    )

    domain_cardinality.append({
        "Domain": domain,
        "Users_Unique": user_count,
        "Items_Unique": item_count,
        "Verified_Pct": verified_pct,
    })


cardinality_by_domain_df = pd.DataFrame(cardinality_by_domain)

print("\nDomain-level Cardinality Summary:")
display(domain_cardinality_df)

# Export
cardinality_comp_df.to_csv('reports/eda/01_cardinality_by_domain.csv', index=False)
print("Saved: reports/eda/01_cardinality_by_domain.csv")


**2.10 DOMAIN COMPARISON VISUALIZATION**

In [ ]:
print("\n" + "="*80)
print("CREATING DOMAIN COMPARISON VISUALIZATIONS")
print("="*80)

if COL_RATING in df.columns:
    # Domain-specific colors (color-blind safe)
    DOMAIN_COLORS = {
        'All_Beauty': '#E69F00',  # Orange
        'Electronics': '#56B4E9', # Sky Blue  
        'Books': '#009E73'        # Bluish Green
    }
    
    # Rating distribution by domain (3-panel figure)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, domain in enumerate(CATEGORIES):
        if domain in df['domain'].values:
            df_domain = df[df['domain'] == domain]
            
            # Get domain color
            domain_color = DOMAIN_COLORS.get(domain, '#0077BB')
            
            # Histogram
            axes[idx].hist(df_domain[COL_RATING], bins=5, 
                          color=domain_color, alpha=0.7, edgecolor='black', linewidth=1.5)
            
            # Title and labels
            axes[idx].set_title(f'{domain}\nRating Distribution', 
                               fontsize=12, fontweight='bold')
            axes[idx].set_xlabel('Rating', fontsize=11)
            axes[idx].set_ylabel('Frequency', fontsize=11)
            axes[idx].set_xticks([1, 2, 3, 4, 5])
            axes[idx].grid(alpha=0.3, linestyle='--')
            
            # Statistics
            mean_rating = df_domain[COL_RATING].mean()
            median_rating = df_domain[COL_RATING].median()
            std_rating = df_domain[COL_RATING].std()
            
            # Add mean line
            axes[idx].axvline(mean_rating, color='#CC3311', # Red
                             linestyle='--', linewidth=2, 
                             label=f'Mean: {mean_rating:.2f}')
            
            # Add stats box
            stats_text = f'Mean: {mean_rating:.2f}\nMedian: {median_rating:.1f}\nStd: {std_rating:.2f}'
            axes[idx].text(0.02, 0.98, stats_text,
                          transform=axes[idx].transAxes,
                          fontsize=9,
                          verticalalignment='top',
                          bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            axes[idx].legend(loc='upper right')
    
    add_panel_labels(axes, ['A', 'B', 'C'])
    plt.tight_layout()
    save_publication_figure(fig, '01_rating_distribution_by_domain')
    plt.show()
    

In [ ]:
# Rating distribution - Stacked bar chart
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data for stacked bar
domains = ['OVERALL'] + CATEGORIES
rating_data = {1: [], 2: [], 3: [], 4: [], 5: []}

for domain_name in domains:
    if domain_name == 'OVERALL':
        df_temp = df
    else:
        df_temp = df[df['domain'] == domain_name]
    
    for rating in [1, 2, 3, 4, 5]:
        count = (df_temp[COL_RATING] == rating).sum()
        pct = (count / len(df_temp)) * 100
        rating_data[rating].append(pct)

# Create stacked bar
x = np.arange(len(domains))
width = 0.6
bottom = np.zeros(len(domains))


colors = ['#CC3311', '#EE7733', '#CCBB44', '#228833', '#0077BB']  # Red, Orange, Yellow, Green, Blue


for i, (rating, values) in enumerate(rating_data.items()):
    ax.bar(x, values, width, label=f'{rating}', bottom=bottom, 
           color=colors[i], alpha=0.8, edgecolor='white', linewidth=2)
    bottom += values

ax.set_xlabel('Domain', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax.set_title('Rating Distribution by Domain (Stacked)', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(domains, rotation=45, ha='right')
ax.legend(title='Rating', loc='upper left', bbox_to_anchor=(1, 1))
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_publication_figure(fig, '01_rating_stacked_distribution')
plt.show()


In [ ]:
# CARDINALITY VISUALIZATION (3-panel)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# PANEL A: Unique Counts Bar Chart

# Sort by unique count
cardinality_sorted = cardinality.sort_values('Unique_Count', ascending=True)

axes[0].barh(cardinality_sorted['Column'], 
             cardinality_sorted['Unique_Count'],
             color='#0077BB', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Unique Count', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Column', fontsize=11, fontweight='bold')
axes[0].set_title('Unique Value Counts by Column', fontsize=12, fontweight='bold')
axes[0].set_xscale('log')  # Log scale for better visibility
axes[0].grid(axis='x', alpha=0.3)

# PANEL B: Unique Ratio Scatter

# Color code by cardinality level
colors = []
for ratio in cardinality['Unique_Ratio']:
    if ratio > 0.5:
        colors.append('#CC3311')  # Red - High cardinality
    elif ratio > 0.1:
        colors.append('#EE7733')  # Orange - Medium
    else:
        colors.append('#228833')  # Green - Low

axes[1].scatter(cardinality['Unique_Count'], 
                cardinality['Unique_Ratio'],
                c=colors, s=100, alpha=0.6, edgecolor='black')
axes[1].set_xlabel('Unique Count (log scale)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Unique Ratio', fontsize=11, fontweight='bold')
axes[1].set_title('Cardinality: Count vs Ratio', fontsize=12, fontweight='bold')
axes[1].set_xscale('log')
axes[1].axhline(y=0.5, color='red', linestyle='--', label='High Cardinality (>50%)')
axes[1].axhline(y=0.1, color='orange', linestyle='--', label='Medium (>10%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

# PANEL C: Cardinality Categories Pie Chart

# Categorize columns
high_card = (cardinality['Unique_Ratio'] > 0.5).sum()
medium_card = ((cardinality['Unique_Ratio'] > 0.1) & (cardinality['Unique_Ratio'] <= 0.5)).sum()
low_card = (cardinality['Unique_Ratio'] <= 0.1).sum()

sizes = [low_card, medium_card, high_card]
labels = [f'Low (<10%)\n{low_card} cols', 
          f'Medium (10-50%)\n{medium_card} cols', 
          f'High (>50%)\n{high_card} cols']
colors_pie = ['#228833', '#EE7733', '#CC3311']
explode = (0.05, 0.05, 0.1) # Explode high cardinality

axes[2].pie(sizes, explode=explode, labels=labels, colors=colors_pie,
            autopct='%1.1f%%', shadow=True, startangle=90)
axes[2].set_title('Cardinality Distribution', fontsize=12, fontweight='bold')

add_panel_labels(axes, ['A', 'B', 'C'])
plt.tight_layout()
save_publication_figure(fig, '01_cardinality_analysis')
plt.show()


In [ ]:
# DOMAIN SIZE TREEMAP

fig, ax = plt.subplots(figsize=(12, 8))

# Domain sizes
domain_sizes = []
domain_labels = []
for domain in CATEGORIES:
    if domain in df['domain'].values:
        count = len(df[df['domain'] == domain])
        pct = (count / len(df)) * 100
        domain_sizes.append(count)
        domain_labels.append(f'{domain}\n{count:,} rows\n({pct:.1f}%)')

# Colors
colors_treemap = ['#E69F00', '#56B4E9', '#009E73']

# Create treemap
squarify.plot(sizes=domain_sizes, 
              label=domain_labels, 
              color=colors_treemap,
              alpha=0.7, 
              text_kwargs={'fontsize': 12, 'weight': 'bold'},
              edgecolor='white',
              linewidth=3)

plt.title('Domain Size Comparison (Treemap)', fontsize=14, fontweight='bold')
plt.axis('off')
save_publication_figure(fig, '01_domain_size_treemap')
plt.show()


In [ ]:
# DESCRIPTIVE STATISTICS VIOLIN PLOTS 

# Select numerical columns (exclude domain)
num_cols_viz = [col for col in num_cols if col != 'domain']

if len(num_cols_viz) > 0:
    n_cols_viz = len(num_cols_viz)
    n_rows = (n_cols_viz + 2) // 3  # 3 columns per row
    n_cols_layout = min(3, n_cols_viz)  # Max 3 columns
    
    fig, axes = plt.subplots(n_rows, n_cols_layout, figsize=(18, n_rows * 5))
    
    # FIXED: Properly handle axes array
    if n_rows == 1 and n_cols_layout == 1:
        axes_flat = [axes]
    elif n_rows == 1:
        axes_flat = axes
    elif n_cols_layout == 1:
        axes_flat = axes
    else:
        axes_flat = axes.flatten()
    
    # Domain colors
    domain_colors = ['#E69F00', '#56B4E9', '#009E73']
    
    for idx, col in enumerate(num_cols_viz):
        # Prepare data for violin plot
        df_plot = df[['domain', col]].copy()
        
        # Violin plot
        parts = axes_flat[idx].violinplot(
            [df[df['domain'] == domain][col].dropna() 
             for domain in CATEGORIES if domain in df['domain'].values],
            positions=range(len(CATEGORIES)),
            showmeans=True,
            showmedians=True
        )
        
        # Color violins
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(domain_colors[i])
            pc.set_alpha(0.6)
            pc.set_edgecolor('black')
            pc.set_linewidth(1.5)
        
        # Styling
        axes_flat[idx].set_title(f'{col} Distribution by Domain', 
                                 fontsize=11, fontweight='bold')
        axes_flat[idx].set_ylabel(col, fontsize=10, fontweight='bold')
        axes_flat[idx].set_xticks(range(len(CATEGORIES)))
        axes_flat[idx].set_xticklabels(CATEGORIES, rotation=45, ha='right')
        axes_flat[idx].grid(alpha=0.3, axis='y', linestyle='--')
        
        # Add mean values as text
        for i, domain in enumerate(CATEGORIES):
            if domain in df['domain'].values:
                mean_val = df[df['domain'] == domain][col].mean()
                axes_flat[idx].text(i, mean_val, f'{mean_val:.1f}', 
                                   ha='center', va='bottom', fontsize=8, 
                                   fontweight='bold')
    
    # Hide empty subplots
    for idx in range(len(num_cols_viz), len(axes_flat)):
        axes_flat[idx].axis('off')
    
    plt.tight_layout()
    save_publication_figure(fig, '01_descriptive_statistics_violin')
    plt.show()

In [ ]:
# DESCRIPTIVE STATISTICS BOX PLOTS

# Select numerical columns (exclude domain)
num_cols_viz = [col for col in num_cols if col != 'domain']

if len(num_cols_viz) > 0:
    n_cols_viz = len(num_cols_viz)
    n_rows = (n_cols_viz + 2) // 3  # 3 columns per row
    n_cols_layout = min(3, n_cols_viz)
    
    fig, axes = plt.subplots(n_rows, n_cols_layout, figsize=(18, n_rows * 5))
    
    # Convert to list
    if isinstance(axes, np.ndarray):
        axes_flat = axes.flatten()
    else:
        axes_flat = [axes]
    
    # Domain colors
    domain_palette = {
        'All_Beauty': '#E69F00',
        'Electronics': '#56B4E9',
        'Books': '#009E73'
    }
    
    for idx, col in enumerate(num_cols_viz):
        # Seaborn box plot (much easier!)
        sns.boxplot(data=df, x='domain', y=col, 
                   ax=axes_flat[idx],
                   palette=domain_palette,
                   notch=True,
                   showmeans=True,
                   meanprops={"marker":"o", "markerfacecolor":"white", 
                             "markeredgecolor":"black", "markersize":8})
        
        axes_flat[idx].set_title(f'{col} Distribution by Domain', 
                                fontsize=11, fontweight='bold')
        axes_flat[idx].set_xlabel('Domain', fontsize=10, fontweight='bold')
        axes_flat[idx].set_ylabel(col, fontsize=10, fontweight='bold')
        axes_flat[idx].tick_params(axis='x', rotation=45)
        axes_flat[idx].grid(alpha=0.3, axis='y')
    
    # Hide empty subplots
    for idx in range(len(num_cols_viz), len(axes_flat)):
        axes_flat[idx].axis('off')
    
    plt.tight_layout()
    save_publication_figure(fig, '01_descriptive_statistics_boxplots')
    plt.show()

In [ ]:
# TEMPORAL ANALYSIS - REVIEW VOLUME OVER TIME

if COL_TIMESTAMP in df.columns:
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    
    # PANEL A: Review Volume Over Time (All Domains)
    
    df['year_month'] = df[COL_TIMESTAMP].dt.to_period('M')
    monthly_counts = df.groupby(['year_month', 'domain']).size().reset_index(name='count')
    
    for domain in CATEGORIES:
        if domain in df['domain'].values:
            domain_data = monthly_counts[monthly_counts['domain'] == domain]
            domain_color = {'All_Beauty': '#E69F00', 
                          'Electronics': '#56B4E9', 
                          'Books': '#009E73'}.get(domain, '#0077BB')
            
            axes[0].plot(domain_data['year_month'].astype(str), 
                        domain_data['count'],
                        marker='o', linewidth=2, label=domain,
                        color=domain_color, alpha=0.7)
    
    axes[0].set_xlabel('Month', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Review Count', fontsize=11, fontweight='bold')
    axes[0].set_title('Review Volume Over Time by Domain', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)
    
    # PANEL B: Cumulative Reviews
    
    for domain in CATEGORIES:
        if domain in df['domain'].values:
            domain_data = monthly_counts[monthly_counts['domain'] == domain].copy()
            domain_data['cumulative'] = domain_data['count'].cumsum()
            domain_color = {'All_Beauty': '#E69F00', 
                          'Electronics': '#56B4E9', 
                          'Books': '#009E73'}.get(domain, '#0077BB')
            
            axes[1].plot(domain_data['year_month'].astype(str), 
                        domain_data['cumulative'],
                        linewidth=2, label=domain,
                        color=domain_color, alpha=0.7)
    
    axes[1].set_xlabel('Month', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Cumulative Review Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Cumulative Reviews Over Time', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)
    
    add_panel_labels(axes, ['A', 'B'])
    plt.tight_layout()
    save_publication_figure(fig, '01_temporal_analysis')
    plt.show()
    

**2.11 BASIC PROFILING SUMMARY - JSON EXPORT**

In [ ]:
# Persist a consolidated snapshot of dataset structure and key statistics for reproducibility and downstream analysis

print("\n" + "="*80)
print("EXPORTING COMPREHENSIVE BASIC PROFILE")
print("="*80)

# Comprehensive profile 

basic_profile = {
    'timestamp': datetime.now().isoformat(),
    'multi_domain': True,
    'domains': CATEGORIES,
    
    # Combined dataset 
    'dataset': {
        'rows': int(n_rows),
        'columns': int(n_cols),
        'memory_mb': float(memory_mb),
        'avg_bytes_per_row': float((memory_mb * 1024**2) / n_rows)
    },
    
    # Column types 
    'column_types': {
        'numerical': num_cols,
        'categorical': cat_cols,
        'datetime': datetime_cols if len(datetime_cols) > 0 else []
    },
    
    # Overall cardinality
    'cardinality': {
        'user_count': int(df[COL_USER].nunique()) if COL_USER in df.columns else None,
        'item_count': int(df[COL_ITEM].nunique()) if COL_ITEM in df.columns else None,
    },
    
    # Overall date range 
    'date_range': {
        'min': str(df[COL_TIMESTAMP].min()) if COL_TIMESTAMP in df.columns else None,
        'max': str(df[COL_TIMESTAMP].max()) if COL_TIMESTAMP in df.columns else None,
        'span_days': int((df[COL_TIMESTAMP].max() - df[COL_TIMESTAMP].min()).days) if COL_TIMESTAMP in df.columns else None
    },
    
    # Per-domain statistics (Multi-domain enhancement)
    'domain_statistics': {},
    
    # Loading summary
    'loading_summary': load_stats
}

# Add per-domain statistics
for domain in CATEGORIES:
    if domain in df['domain'].values:
        df_domain = df[df['domain'] == domain]
        
        basic_profile['domain_statistics'][domain] = {
            'rows': int(len(df_domain)),
            'percentage': float(len(df_domain) / n_rows * 100),
            'memory_mb': float(df_domain.memory_usage(deep=True).sum() / 1024**2),
            'user_count': int(df_domain[COL_USER].nunique()) if COL_USER in df_domain.columns else None,
            'item_count': int(df_domain[COL_ITEM].nunique()) if COL_ITEM in df_domain.columns else None,
            'avg_rating': float(df_domain[COL_RATING].mean()) if COL_RATING in df_domain.columns else None,
            'verified_pct': float((df_domain[COL_VERIFIED].sum() / len(df_domain)) * 100) if COL_VERIFIED in df_domain.columns else None,
            'date_range': {
                'min': str(df_domain[COL_TIMESTAMP].min()) if COL_TIMESTAMP in df_domain.columns else None,
                'max': str(df_domain[COL_TIMESTAMP].max()) if COL_TIMESTAMP in df_domain.columns else None
            }
        }

# Export basic profile 
with open('data/processed/01_basic_profile.json', 'w') as f:
    json.dump(basic_profile, f, indent=2, default=str)

print("\nBasic profile saved: data/processed/01_basic_profile.json")


**<font color='green'>SECTION 2 COMPLETE</font>**

In [ ]:
# Section 2 complete
# Multi-domain data loading and basic profiling have been completed. 
print("\n" + "=" * 80)
print("All intermediate artefacts required for reproducibility and downstream analysis have been persisted.")
print("=" * 80)

print("\nPrepared datasets and profiling artefacts are ready.")
print("Proceeding to Section 3: Data Quality & Trustworthiness")
print("=" * 80 + "\n")